In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef, XSD
from elasticsearch import Elasticsearch

from utils import EMBEDD_MODEL_1

INDEX_NAME = os.getenv("INDEX_NAME")

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_kg(df, embed_model):
    EMBEDDING_SIZE = 384
    MAPPING = {
        "properties": {
            "id":   {"type": "integer"},
            "value_name": {"type": "text"},
            "value_type": {"type": "text"},
            "triplet_id": {"type": "integer"},
            "value_embedding": {
                "type": "dense_vector",
                "dims": EMBEDDING_SIZE,
                "index": True,
                "similarity": "cosine",
            }
        }
    }

    TRIPLET_MAPPING = {
        "properties": {
            "triplet_id": {"type": "integer"},
            "embedding": {
                "type": "dense_vector",
                "dims": EMBEDDING_SIZE,
                "index": True,
                "similarity": "cosine",
            }
        }
    }

    TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"
    ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
    es_client = Elasticsearch('http://localhost:9200')
    es_client.indices.delete(index=ENT_INDEX_NAME, ignore=[400, 404])
    es_client.indices.create(index=ENT_INDEX_NAME, mappings=MAPPING)

    es_client.indices.delete(index=TRIPLETS_INDEX_NAME, ignore=[400, 404])
    es_client.indices.create(index=TRIPLETS_INDEX_NAME, mappings=TRIPLET_MAPPING)

    BASE = "https://example.com/political-kg/"

    ENTITY = Namespace(f"{BASE}entity/")
    PREDICATE = Namespace(f"{BASE}predicate/")
    ASSERTION = Namespace(f"{BASE}assertion/")
    PROPERTY = Namespace(f"{BASE}property/")

    graph = Graph()

    graph.bind("entity", ENTITY)
    graph.bind("predicate", PREDICATE)
    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)

    entity_embeddings = {}
    relation_embeddings = {}

    for _, row in tqdm(df.iterrows(), "Creating knowledge graph from triplets", total=len(df)):

        subject = row["subject"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "")
        object_ = row["object"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "")
        predicate = row["predicate"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "")

        subject_label = subject.replace(" ", "_")
        predicate_label = predicate.replace(" ", "_")
        object_label = object_.replace(" ", "_")

        speech_id = row["speech_id"]
        fragment_start = row["start"]
        fragment_end = row["end"]
        fragment_date = row["date"]

        subject_uri = URIRef(f"{ENTITY}{subject_label}")
        predicate_uri = URIRef(f"{PREDICATE}{predicate_label}")
        object_uri = URIRef(f"{ENTITY}{object_label}")

        statement = URIRef(f"{ASSERTION}/{speech_id}_{fragment_start}_{fragment_end}_{np.random.randint(100000)}")

        if subject not in entity_embeddings:
            embedding = embed_model.encode(
                subject,
                normalize_embeddings=True
            )
            entity_embeddings[subject] = embedding

        if object_ not in entity_embeddings:
            embedding = embed_model.encode(
                object_,
                normalize_embeddings=True
            )
            entity_embeddings[object_] = embedding


        if predicate not in relation_embeddings:
            embedding = embed_model.encode(
                predicate,
                normalize_embeddings=True
            )

            relation_embeddings[predicate] = embedding

        sub_embedding = entity_embeddings[subject]
        obj_embedding = entity_embeddings[object_]
        pred_embedding = relation_embeddings[predicate]

        triplet_embedding = np.average(
            [sub_embedding, obj_embedding, pred_embedding],
            axis=0,
            weights=[0.4, 0.2, 0.4]
        )
        triplet_doc = {
            "id": hash(statement) % (10 ** 8),
            "triplet_id": hash(statement) % (10 ** 8),
            "embedding": triplet_embedding.tolist()
        }
        es_client.index(index=TRIPLETS_INDEX_NAME, id=triplet_doc["triplet_id"], document=triplet_doc)
        graph.add((statement, RDF.type, RDF.Statement))
        graph.add((statement, RDF.subject, subject_uri))
        graph.add((statement, RDF.predicate, predicate_uri))
        graph.add((statement, RDF.object, object_uri))
        graph.add((statement, PROPERTY.speech_id, Literal(speech_id)))
        graph.add((statement, PROPERTY.start, Literal(int(fragment_start))))
        graph.add((statement, PROPERTY.end, Literal(int(fragment_end))))
        graph.add((statement, PROPERTY.date, Literal(fragment_date, datatype=XSD.date)))
        graph.add((statement, PROPERTY.triplet_id, Literal(triplet_doc["triplet_id"], datatype=XSD.integer)))

    for entity, embedding in tqdm(entity_embeddings.items(), "Indexing entities in Elasticsearch"):
        entity_doc = {
            "id": hash(entity) % (10 ** 8),
            "value_name": entity.replace(" ", "_"),
            "value_type": "entity",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=entity_doc["id"], document=entity_doc)

    for relation, embedding in tqdm(relation_embeddings.items(), "Indexing relations in Elasticsearch"):
        relation_doc = {
            "id": hash(relation) % (10 ** 8),
            "value_name": relation.replace(" ", "_"),
            "value_type": "relation",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=relation_doc["id"], document=relation_doc)
    return graph

In [3]:
new_triplets = pd.read_csv("data/new_triplets.csv")

In [4]:
embed_model = EMBEDD_MODEL_1

g = create_kg(new_triplets, embed_model)
g.serialize(destination=f"{INDEX_NAME}.ttl", format="turtle")

/tmp/ipykernel_43420/985728516.py:33: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.delete(index=ENT_INDEX_NAME, ignore=[400, 404])
/tmp/ipykernel_43420/985728516.py:36: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.delete(index=TRIPLETS_INDEX_NAME, ignore=[400, 404])
Indexing relations in Elasticsearch: 100%|██████████| 5631/5631 [00:35<00:00, 157.86it/s]


<Graph identifier=N3a00bcd43a994b8ca316fa21640451ff (<class 'rdflib.graph.Graph'>)>